
# DESC SN Ia metric # 



**What this notebook does (quick sanity-check run of `SNNSNMetric`)**

`SNNSNMetric` (contributed by the DESC/Philippe Gris group) is a *cadence metric*: instead of just
counting visits, it asks a survey-strategy question — "given the actual sequence of LSST visits
(nights, filters, depths) at a given point on the sky, how well could we discover and characterize
Type Ia supernovae there?"

For each sky pixel it internally does the following, using ONLY the simulated visits that fall in that pixel:
1. Generate a grid of fake SNe Ia at different redshifts and explosion (peak) dates.
2. For each fake SN, check which of the real LSST visits would have observed it before/after peak
   brightness, and estimate the photometric uncertainty of each point (using a gamma/noise model).
3. Fit a simplified SALT2-like light curve and evaluate the color-uncertainty (sigma_color).
4. Find `zlim`: the highest redshift at which enough SNe pass quality cuts (sigma_color threshold,
   minimum epochs before/after peak) to be usable for cosmology.
5. Combine `zlim` with the SN Ia volumetric rate to get `nSN`: the expected number of well-measured
   SNe Ia out to that redshift, for that field/pixel and survey duration.

This notebook runs the metric with *default parameters* on a coarse HEALPix grid (`nside=4`, i.e. only
192 pixels over the whole sky) purely to check that the pipeline runs end-to-end quickly, not to get a
science-quality result (for that, see `01_testSNIa.ipynb` and `02_Number_SNeIa_metric.ipynb`).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

import healpy as hp
import pandas as pd

import rubin_sim.maf as maf
from rubin_sim.data import get_baseline
import time

## Configuration

In [ ]:
# RUBIN_SIM_DATA_DIR points to the local cache of rubin_sim/rubin_scheduler auxiliary data
# (baseline opsim database, dust maps, throughputs, SN gamma files, etc.).
# os.environ["RUBIN_SIM_DATA_DIR"] = "/users/dagoret/DATA/OpSim"
path_topdir = os.getenv("RUBIN_SIM_DATA_DIR")
print(f"path_topdir = {path_topdir}")

In [ ]:
# Baseline Survey
# get_baseline() locates the officially recommended LSST baseline cadence simulation
# (an opsim .db file containing the full list of simulated visits: time, filter, sky position, depth...).
# This is the "survey strategy" the metric will be evaluated against.
baseline_file = get_baseline()
runName = os.path.split(baseline_file)[-1].replace(".db", "")

print(runName)

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    # Scratch directory where MAF will write its SQLite results database and any output products
    # for this run; a fresh temporary directory avoids clobbering results from other notebooks.
    data_dir_itself = tempfile.TemporaryDirectory(prefix="00_maf_SNIa_FastNSIDE4", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

In [ ]:
# Set up MAF output
# ResultsDb is MAF's small SQLite database that indexes every metric bundle that gets run/plotted,
# so results can be tracked and reloaded later (e.g. via showMaf).
outDir = data_dir
resultsDb = maf.db.ResultsDb(out_dir=outDir)

## Define Slicer, SNNSNMetric and summary

In [ ]:
# Inspect the signature/docstring of MetricBundleGroup (the object that actually executes
# a metric against the opsim database and dispatches visits to each slice point).
%pinfo maf.MetricBundleGroup

In [ ]:
# Plotting options for the resulting sky maps: clip color scale at the 95th percentile
# so a few outlier pixels don't wash out the color range, and use 5 ticks on the color bar.
plot_dict = {"percentile_clip": 95.0, "n_ticks": 5}

# HEALPix resolution: nside=4 -> 12*4^2 = 192 pixels over the whole sky (very coarse, fast to run).
# Increase nside for finer angular resolution at the cost of runtime (metric is evaluated per pixel).
sne_nside = 4

# Summary metrics collapse the per-pixel metric_values array (a skymap) down to single numbers:
# median and mean over all pixels, and a sum (here used to get "Total detected" SNe over the sky).
sn_summary = [maf.MedianMetric(), maf.MeanMetric(), maf.SumMetric(metric_name="Total detected")]

# HealpixSlicer partitions the sky into equal-area HEALPix pixels; for each pixel it will select
# only the simulated visits whose pointing overlaps that pixel and hand them to the metric.
slicer = maf.HealpixSlicer(nside=sne_nside, use_cache=False)
# Alternative: HealpixSubsetSlicer restricts evaluation to a handful of chosen pixel indices (hpid),
# useful for fast debugging on a single line of sight instead of the whole sky.
# slicer = maf.HealpixSubsetSlicer(nside=16, hpid=[890], useCache=False)
# slicer = maf.HealpixSubsetSlicer(nside=16, hpid=[889], useCache=False)

# The cadence metric itself, run here with all default parameters (see 01_testSNIa.ipynb for an
# example that sets n_bef/n_aft/zmin/zmax/etc. explicitly and explains what each one controls).
metric = maf.SNNSNMetric(verbose=False)

# A MetricBundle packages together: the metric, the slicer, an (empty here) SQL constraint on which
# visits to use, plotting options, and the summary metrics -- everything MAF needs to run and plot it.
bundle = maf.MetricBundle(
    metric, slicer, None, plot_dict=plot_dict, summary_metrics=sn_summary, run_name=runName
)

# A MetricBundleGroup ties one or more bundles to a specific opsim database (baseline_file) and
# output location; it handles querying the visit database and looping the metric over all slice points.
bg = maf.MetricBundleGroup({"sn": bundle}, baseline_file, outDir, resultsDb)

In [ ]:
# run_all() queries the opsim database, runs SNNSNMetric on every HEALPix pixel, and computes the
# summary metrics; plot_all() then renders the resulting sky map(s) (nSN, zlim, ...).
t1 = time.time()
bg.run_all()
t2 = time.time()
bg.plot_all(closefigs=False)
print("runtime=", t2 - t1, "s")

In [ ]:
# Raw per-pixel metric output (a numpy masked array with one entry per HEALPix pixel).
# .compressed() drops masked (e.g. never-observed) pixels and returns only valid values.
bundle.metric_values.compressed()

In [ ]:
# The 'reduce' values of the metric got stored in the bundle dict in the bungle group
# SNNSNMetric returns a compound result per pixel; MAF's "reduce" mechanism splits it into
# separate named bundles (one per reduce_* method), stored here in bg.bundle_dict.
bg.bundle_dict

In [ ]:
# The nSN and zlim values are pulled out in those reduce methods, into their own bundles.
# "SNNSNMetric_reducen_sn": expected number of well-measured SNe Ia per pixel (survey duration).
# "SNNSNMetric_reducezlim": the redshift completeness limit per pixel (see markdown cell above).
bdict = bg.bundle_dict
print(bdict["SNNSNMetric_reducen_sn"].metric_values.compressed())
np.median(bdict["SNNSNMetric_reducen_sn"].metric_values.compressed())

In [ ]:
# Redshift completeness limit (zlim) per pixel -- the SN Ia cosmology reach of the cadence.
bdict["SNNSNMetric_reducezlim"].metric_values.compressed()